# Embeddings

This lab is designed to help you solidify your understanding of embeddings by applying them to tasks like semantic similarity, clustering, and building a semantic search system.

### Tasks:
- Task 1: Semantic Similarity Comparison
- Task 2: Document Clustering
- Task 3: Enhance the Semantic Search System


## Task 1: Semantic Similarity Comparison
### Objective:
Compare semantic similarity between pairs of sentences using cosine similarity and embeddings.

### Steps:
1. Load a pre-trained Sentence Transformer model.
2. Encode the sentence pairs.
3. Compute cosine similarity for each pair.

### Dataset:
- "A dog is playing in the park." vs. "A dog is running in a field."
- "I love pizza." vs. "I enjoy ice cream."
- "What is AI?" vs. "How does a computer learn?"


In [17]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")

# Load pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sentence pairs
sentence_pairs = [
    ("A dog is playing in the park.", "A dog is running in a field."),
    ("I love pizza.", "I enjoy ice cream."),
    ("What is AI?", "How does a computer learn?")
]

# Compute similarities

#YOUR CODE HERE
#Encode each pair
for i, (sent1, sent2) in enumerate(sentence_pairs):
    embedding1= model.encode([sent1])
    embedding2= model.encode([sent2])
    
    # Cosine similarity
    similarities= cosine_similarity(embedding1, embedding2)
    
    # Print results 
    print(f"Pair {i+1}:")
    print(f"  Sentence 1: {sent1}")
    print(f"  Sentence 2: {sent2}")
    print(f"  Similarity Score: {similarities}")
  

Pair 1:
  Sentence 1: A dog is playing in the park.
  Sentence 2: A dog is running in a field.
  Similarity Score: [[0.5219753]]
Pair 2:
  Sentence 1: I love pizza.
  Sentence 2: I enjoy ice cream.
  Similarity Score: [[0.52806807]]
Pair 3:
  Sentence 1: What is AI?
  Sentence 2: How does a computer learn?
  Similarity Score: [[0.31943488]]


### Questions:
- Which sentence pairs are the most semantically similar? Why?
- Can you think of cases where cosine similarity might fail to capture true semantic meaning?


In [ ]:
# Pair 2 is the most simialar (0.5281), both express positive sentiment about food with similar structure.
#Sarcasm: "Great job!" (sincere vs sarcastic)
#Negation: "I love pizza" vs "I don't love pizza"
#Context: "Bank" (financial vs river)

## Task 2: Document Clustering
### Objective:
Cluster a set of text documents into similar groups based on their embeddings.

### Steps:
1. Encode the documents using Sentence Transformers.
2. Use KMeans clustering to group the documents.
3. Analyze the clusters for semantic meaning.

In [18]:
from sklearn.cluster import KMeans

# Documents to cluster
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?"
]

# Encode documents

#YOUR CODE HERE

doc_embeddings = model.encode(documents)

In [33]:
# Perform KMeans clustering

#YOUR CODE HERE
kmeans = KMeans(n_clusters=2, random_state=42)

#Fit the model
kmeans.fit(doc_embeddings)

#Predict cluster labels
cluster_labels = kmeans.fit_predict(doc_embeddings)

print(f"Cluster labels: {cluster_labels}")

Cluster labels: [0 0 0 1 1 1]


In [34]:
# Print cluster assignments
from collections import defaultdict

#YOUR CODE HERE

clusters = defaultdict(list)
for doc, label in zip(documents, cluster_labels):
    clusters[label].append(doc)

for cluster_id, docs in clusters.items():
    print(f"Cluster {cluster_id}: {docs}")


Cluster 0: ['What is the capital of France?', 'How do I bake a chocolate cake?', 'What is the distance between Earth and Mars?']
Cluster 1: ['How do I change a flat tire on a car?', 'What is the best way to learn Python?', 'How do I fix a leaky faucet?']


### Questions:
- How many clusters make the most sense? Why?
- Examine the documents in each cluster. Are they semantically meaningful? Can you assign a semantic "theme" to each cluster?
- Try this exercise with a larger dataset of your choice

In [ ]:
# Number of clusters: 2 clusters provide the clearest separation.
# Cluster 0: Knowledge/factual questions (e.g., capital of France, distance to Mars, baking a cake). Some topics are mixed but broadly factual.
# Cluster 1: Practical “how-to” tasks and learning questions (e.g., fixing a tire, learning Python, fixing a faucet).
# Semantic meaning: Cluster 1 is more coherent (practical tasks), while Cluster 0 is somewhat mixed. 
# Overall, 2 clusters capture broad themes effectively.

In [53]:
#Try with a larger dataset

from datasets import load_dataset

dataset = load_dataset("ag_news", split='train[:200]')
documents = [item['text'] for item in dataset]

In [62]:
# Encode document

doc_embeddings = model.encode(documents)

# Perform KMeans clustering

kmeans = KMeans(n_clusters=7, random_state=42)

#Fit the model
kmeans.fit(doc_embeddings)

#Predict cluster labels
cluster_labels = kmeans.fit_predict(doc_embeddings)

print(f"Cluster labels: {cluster_labels}")


Cluster labels: [5 5 2 2 2 2 2 2 0 5 2 2 2 5 2 0 0 2 2 2 5 2 2 2 2 2 5 0 3 2 3 4 2 0 3 4 5
 0 4 4 1 2 0 0 0 0 2 0 0 0 1 4 0 2 4 0 0 3 4 5 0 3 3 3 3 3 1 4 4 4 0 5 5 0
 0 0 5 5 0 4 0 4 4 0 0 4 5 4 4 1 1 6 6 3 1 3 5 5 5 6 6 6 6 1 6 6 1 6 1 4 1
 6 1 1 6 1 6 6 1 6 3 6 6 1 6 0 6 1 6 3 6 3 1 1 1 6 1 1 1 6 1 1 1 1 1 1 1 1
 1 1 1 1 6 1 1 1 6 6 1 0 4 0 5 4 3 2 4 0 4 4 4 3 4 3 4 4 3 4 4 4 5 5 5 0 4
 5 4 1 1 1 0 0 0 0 0 0 0 6 3 4]


In [ ]:
#Finding optimal cluster number
from sklearn.metrics import silhouette_score

for k in range(2, 10):
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(doc_embeddings)
    score = silhouette_score(doc_embeddings, labels)
    print(f'k={k}, Silhouette Score={score:.3f}')

k=2, Silhouette Score=0.030
k=3, Silhouette Score=0.029
k=4, Silhouette Score=0.030
k=5, Silhouette Score=0.025
k=6, Silhouette Score=0.023
k=7, Silhouette Score=0.028
k=8, Silhouette Score=0.021
k=9, Silhouette Score=0.023


In [ ]:
# Choosing 7 clusters based on Silhouette Score
# Based on the Silhouette scores for k=2 to k=9, k=7 gives a good balance between cluster separation and cohesion.
# Therefore, we select 7 clusters to capture distinct themes in the dataset while avoiding too much overlap.

In [ ]:
#Creating a dataframe from news articles and cluster assignments
import pandas as pd

#Create a dataframe for documents
all_news = pd.DataFrame(documents, columns=['news'])

# Add the cluster labels
clusters_df = pd.concat([all_news, pd.DataFrame(cluster_labels, columns = ['cluster'])], axis =1)

clusters_df.head()


,news,cluster
0,Wall St. Bears Claw Back Into the Black (Reute...,5
1,Carlyle Looks Toward Commercial Aerospace (Reu...,5
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [69]:
# Print cluster assignments
# See all documents by cluster
for cluster_id, group in clusters_df.groupby('cluster'):
    print(f"\nCluster {cluster_id}:")
    print(group['news'].head(5)) # print first 5 documents of each cluster



Cluster 0:
8     Safety Net (Forbes.com) Forbes.com - After ear...
15    Rescuing an Old Saver If you think you may nee...
16    Kids Rule for Back-to-School The purchasing po...
27    HP shares tumble on profit news Hewlett-Packar...
33    Saudi phone sector gets \$1bn lift A group led...
Name: news, dtype: object

Cluster 1:
40    Letters Target the abusers of legal weapons We...
50    Making Your Insurer Pay If Hurricane Charley b...
66    IT alligator tales I grew up in New York, wher...
89    Charley's Force Took Experts by Surprise (AP) ...
90    Science, Politics Collide in Election Year (AP...
Name: news, dtype: object

Cluster 2:
2    Oil and Economy Cloud Stocks' Outlook (Reuters...
3    Iraq Halts Oil Exports from Main Southern Pipe...
4    Oil prices soar to all-time record, posing new...
5    Stocks End Up, But Near Year Lows (Reuters) Re...
6    Money Funds Fell in Latest Week (AP) AP - Asse...
Name: news, dtype: object

Cluster 3:
28    Mauritian textile firm cuts jobs 

## Task 3: Semantic Search System
### Objective:
Create a semantic search engine:
A user provides a query and you search the dataset for semantically relevant documents to return. Return the top 5 results.

### Dataset:
- Use the following set of documents:
    - "What is the capital of France?"
    - "How do I bake a chocolate cake?"
    - "What is the distance between Earth and Mars?"
    - "How do I change a flat tire on a car?"
    - "What is the best way to learn Python?"
    - "How do I fix a leaky faucet?"
    - "What are the best travel destinations in Europe?"
    - "How do I set up a local server?"
    - "What is quantum computing?"
    - "How do I build a mobile app?"


In [35]:
import numpy as np

# Documents dataset
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?",
    "What are the best travel destinations in Europe?",
    "How do I set up a local server?",
    "What is quantum computing?",
    "How do I build a mobile app?"
]

# Compute document embeddings

#YOUR CODE HERE
doc_embeddings= model.encode(documents)

In [43]:
# Create the search function
#This function should encode the user query and return the top N documents that most resemble it
def semantic_search(query, documents, doc_embeddings, top_n=5):
    # YOUR CODE HERE
    query_embedding = model.encode([query])

    #compute cosine similarity between query and documents
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

    # Get the indices of the top N most similar documents
    top_indices = similarities.argsort()[::-1][:top_n]

    # Return the top N documents with similarity scores
    results = [(documents[i], similarities[i]) for i in top_indices]
    return results

In [47]:
# Test the search function
query = "Explain programming languages."
top_results = semantic_search(query, documents, doc_embeddings)


In [49]:
# Print results
for i, (doc, score) in enumerate(top_results):
    print(f"{i+1}. {doc} (Similarity: {score:.3f})")

1. What is quantum computing? (Similarity: 0.435)
2. What is the best way to learn Python? (Similarity: 0.319)
3. How do I build a mobile app? (Similarity: 0.110)
4. How do I set up a local server? (Similarity: 0.091)
5. What are the best travel destinations in Europe? (Similarity: 0.091)


### Questions:
- What are the top-ranked results for the given queries?
- How can you improve the ranking explanation for users?
- Try this approach with a larger dataset

In [ ]:
#Top-ranked results for the query "Explain programming languages:
# 1. "What is quantum computing?" (Similarity: 0.435)
# 2. "What is the best way to learn Python?" (Similarity: 0.319)
# 3. "How do I build a mobile app?" (Similarity: 0.110)
# 4. "How do I set up a local server?" (Similarity: 0.091)
# 5. "What are the best travel destinations in Europe?" (Similarity: 0.091)


#Improving ranking explanation for users:
# Show the similarity score alongside each result to indicate relevance.
# Categorize results (e.g., “Programming concepts” vs. “Practical tasks”) to make the output easier to interpret.
# Highlight keywords or concepts that overlap with the query to give context for the ranking.

In [70]:
# Try with a larger dataset

from datasets import load_dataset

dataset = load_dataset("ag_news", split='train[:200]')
documents = [item['text'] for item in dataset]

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Compute document embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(documents, convert_to_numpy=True)

In [76]:
# Create the search function for larger dataset
#This function should encode the user query and return the top N documents that most resemble it
def semantic_search(query, documents, doc_embeddings, top_n=5):
    # YOUR CODE HERE
    query_embedding = model.encode([query], convert_to_numpy=True)

    #compute cosine similarity between query and documents
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

    # Get the indices of the top N most similar documents
    top_indices = similarities.argsort()[::-1][:top_n]

    # Return the top N documents with similarity scores
    results = [(documents[i], similarities[i]) for i in top_indices]
    return results

In [87]:
# Test the search function
query = "Space missions and astronauts"
top_results = semantic_search(query, documents, doc_embeddings)

In [ ]:
# Convert results to DataFrame
df = pd.DataFrame(top_results, columns=["Document", "Similarity"])
df.index += 1  # Start numbering at 1


In [86]:
# Print results
# for i, (doc, score) in enumerate(top_results):
#     print(f"{i+1}. {doc} (Similarity: {score:.3f})")

df

,Document,Similarity
1,Space Science Pioneer Van Allen Questions Huma...,0.507643
2,Redesigning Rockets: NASA Space Propulsion Fin...,0.503658
3,China Begins Manned Space Flight Countdown (AP...,0.444157
4,New NASA Supercomputer to Aid Theorists and Sh...,0.398079
5,Website Lets Users Scout the Red Planet from H...,0.390900
